# Deep Past Initiative — Notebook submission

This notebook runs on **Kaggle**. Kaggle will re-run it with a **hidden test set** and use the **output file** you select as your submission.

**You do NOT upload a CSV.** The notebook **creates** `submission.csv` when you run it (it writes to `/kaggle/working/`). The competition data (including test) is already provided as **Input** by Kaggle.

**Inputs (add both):** **Competition dataset** (has test.csv) and **your model dataset** (e.g. finetuned-nllb or base NLLB). Right panel → **+ Add input** → add the competition (search "deep past") and your model dataset.

**Steps:**
1. **Run All** (run every cell). The last cell writes `submission.csv` to `/kaggle/working/`.
2. **Find the output:** In the **right panel**, open the **Data** / **Output** section. After a run, files in `/kaggle/working/` (including `submission.csv`) appear there. If you don't see "Output", try **Save Version** → **Save & Run All (Commit)** first; then open that **Version** and check the **Output** tab for the generated files.
3. **Submit:** From the competition page, go to **Submit Predictions** (or **Code** → your notebook). Choose **Notebook** as submission type, select this notebook and the **saved Version** that ran successfully, then select **submission.csv** as the output file to submit.

The CSV must have columns: **id**, **translation** (one row per test sample).

## 1. Paths and config

On Kaggle, competition data is under `/kaggle/input/<competition-slug>/`. We write the submission to `/kaggle/working/` so it appears in Output.

In [ ]:
import os
import pandas as pd

# Kaggle: competition data path (may vary when re-run for submission)
COMPETITION_SLUG = "deep-past-initiative-machine-translation"
INPUT_DIR = f"/kaggle/input/{COMPETITION_SLUG}"
OUTPUT_DIR = "/kaggle/working"

# If default path does not exist, search for test.csv under /kaggle/input
if not os.path.isfile(os.path.join(INPUT_DIR, "test.csv")):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "config.json" in files:
            dirs.clear()
            continue
        if "test.csv" in files:
            INPUT_DIR = root
            break

# Column names (must match competition Data/Evaluation tabs)
TEST_ID_COL = "id"
TEST_SOURCE_COL = "transliteration"
SUBMISSION_ID_COL = "id"
SUBMISSION_PRED_COL = "translation"

# If running locally (e.g. data/raw), uncomment:
# INPUT_DIR = "../data/raw"
# OUTPUT_DIR = "../data/submissions"

print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)
print("Files in input:", os.listdir(INPUT_DIR) if os.path.isdir(INPUT_DIR) else "(dir not found)")

## 2b. Load NLLB model (optional; no fine-tune)

**On Kaggle (no internet):** Add a **Kaggle Dataset** that contains the NLLB model (saved with `save_pretrained`). Use the script `save_nllb_for_kaggle.py` locally once, then zip the folder and create a new Dataset; add it as an **Input** to this notebook. The code below looks for any input that is not the competition and contains `config.json` (e.g. your NLLB dataset).

**If no model input is found,** the notebook falls back to a placeholder so it still runs and produces `submission.csv`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # 0=all, 1=no INFO, 2=no WARNING, 3=no ERROR

# Find model path: search under /kaggle/input for any dir containing config.json (recursive)
# Kaggle can mount datasets at e.g. /kaggle/input/datasets/username/dataset-name/nllb-200-distilled-600M/
MODEL_PATH = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if COMPETITION_SLUG in dirs:
            dirs.remove(COMPETITION_SLUG)  # don't descend into competition data
        if "config.json" in files:
            MODEL_PATH = root
            break

# NLLB language codes (Akkadian not in NLLB; we use Arabic as proxy for Semitic → English)
SRC_LANG = "arb_Arab"   # Arabic (proxy); try "eng_Latn" if transliteration is Latin-script
TGT_LANG = "eng_Latn"
BATCH_SIZE = 8

# Load model and tokenizer directly (avoids pipeline "list has no attribute keys" with local path)
model_nllb = None
tokenizer_nllb = None
if MODEL_PATH:
    try:
        from transformers import AutoModelForSeq2SeqLM
        import torch
        tokenizer_json = os.path.join(MODEL_PATH, "tokenizer.json")
        if os.path.isfile(tokenizer_json):
            from transformers import PreTrainedTokenizerFast
            tokenizer_nllb = PreTrainedTokenizerFast(tokenizer_file=tokenizer_json)
        else:
            try:
                from transformers import AutoTokenizer
                tokenizer_nllb = AutoTokenizer.from_pretrained(MODEL_PATH)
            except Exception:
                from transformers import NllbTokenizer
                tokenizer_nllb = NllbTokenizer.from_pretrained(MODEL_PATH)
        if tokenizer_nllb.pad_token is None:
            tokenizer_nllb.pad_token = tokenizer_nllb.eos_token
        model_nllb = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
        device = 0 if torch.cuda.is_available() else -1
        if device >= 0:
            model_nllb = model_nllb.to(device)
        print("Loaded NLLB from", MODEL_PATH)
    except Exception as e:
        import traceback
        print("Model load failed:", e)
        traceback.print_exc()
        model_nllb = None
        tokenizer_nllb = None
if model_nllb is None:
    print("Using placeholder translator (add NLLB dataset as input to use the model).")

## 2. Load test set

We load the test CSV. During re-run, Kaggle replaces the public test with the hidden test; the columns stay the same (**id**, **transliteration**).

In [ ]:
test_path = os.path.join(INPUT_DIR, "test.csv")
if not os.path.isfile(test_path):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "config.json" in files:
            dirs.clear()
            continue
        if "test.csv" in files:
            test_path = os.path.join(root, "test.csv")
            INPUT_DIR = root
            break
if not os.path.isfile(test_path):
    raise FileNotFoundError("test.csv not found. Add the competition dataset: right panel → + Add input → search 'deep past initiative' or the competition name → Add.")
test = pd.read_csv(test_path, encoding="utf-8", on_bad_lines="warn")

id_col = TEST_ID_COL if TEST_ID_COL in test.columns else test.columns[0]
src_col = TEST_SOURCE_COL if TEST_SOURCE_COL in test.columns else test.columns[1]

sources = test[src_col].astype(str).tolist()
print(f"Test rows: {len(test)}")
print(test.head())

## 3. Translate

Uses the NLLB pipeline from step 2b if loaded; otherwise placeholder. Batched inference for speed.

In [ ]:
def translate(sources):
    if model_nllb is None or tokenizer_nllb is None:
        return ["Translated text placeholder."] * len(sources)
    import torch
    # Ensure pad token is set (required when loading from tokenizer.json only)
    if tokenizer_nllb.pad_token is None or getattr(tokenizer_nllb, "pad_token_id", None) is None or tokenizer_nllb.pad_token_id < 0:
        if getattr(tokenizer_nllb, "eos_token", None) is not None:
            tokenizer_nllb.pad_token = tokenizer_nllb.eos_token
            tokenizer_nllb.pad_token_id = tokenizer_nllb.eos_token_id
        else:
            tokenizer_nllb.add_special_tokens({"pad_token": "[PAD]"})
            model_nllb.resize_token_embeddings(len(tokenizer_nllb))
    device = next(model_nllb.parameters()).device
    if hasattr(tokenizer_nllb, "src_lang"):
        tokenizer_nllb.src_lang = SRC_LANG
    forced_bos_id = tokenizer_nllb.convert_tokens_to_ids(TGT_LANG)
    out = []
    for i in range(0, len(sources), BATCH_SIZE):
        batch = sources[i : i + BATCH_SIZE]
        inputs = tokenizer_nllb(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items() if k in ("input_ids", "attention_mask")}
        with torch.no_grad():
            gen = model_nllb.generate(**inputs, forced_bos_token_id=forced_bos_id, max_length=256)
        out.extend(tokenizer_nllb.batch_decode(gen, skip_special_tokens=True))
    return out

predictions = translate(sources)
print(f"Translated {len(predictions)} segments.")

## 4. Build and save submission.csv

We write **submission.csv** to `/kaggle/working/`. This is the file you must **Add as output** and then **Submit to Competition**.

In [ ]:
submission = pd.DataFrame({
    SUBMISSION_ID_COL: test[id_col],
    SUBMISSION_PRED_COL: predictions,
})

out_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(out_path, index=False, encoding="utf-8")
print(f"Saved {out_path} with {len(submission)} rows.")
print(submission.head())